# 🧠 Rotterdam CT Score Calculator — Demo

**Pipeline:** Binary U-Net + HU Kural Sistemi + Qi et al. 2013 (MLS) + Toledo et al. 2021 (Basal)

**Kullanım:**
1. Tüm hücreleri sırayla çalıştır
2. Son hücrede Gradio URL'si çıkar
3. URL'ye tarayıcıdan gir, DICOM dosyası yükle

## Hücre 1 — Kurulum

In [1]:
import subprocess, sys

def pip(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {pkg}")

pip("pydicom")
pip("scikit-image")
pip("scipy")
pip("segmentation-models-pytorch")
pip("pylibjpeg")
pip("pylibjpeg-libjpeg")
pip("pylibjpeg-openjpeg")
pip("gradio")

# gdcm
r = subprocess.run([sys.executable, "-m", "pip", "install", "gdcm", "-q"],
                   capture_output=True, text=True)
print(f"{'✅' if r.returncode==0 else '⚠️ '} gdcm")

import pydicom
pydicom.config.pixel_data_handlers = ["gdcm", "pylibjpeg", "numpy"]

print("\n✅ Kurulum tamamlandı")

✅ pydicom
✅ scikit-image
✅ scipy
✅ segmentation-models-pytorch
✅ pylibjpeg
✅ pylibjpeg-libjpeg
✅ pylibjpeg-openjpeg
✅ gradio
✅ gdcm

✅ Kurulum tamamlandı


## Hücre 2 — Import'lar

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from pathlib import Path
from collections import defaultdict
from typing import Tuple, List

import pydicom
from scipy import ndimage
from skimage.morphology import (
    remove_small_objects, binary_closing, disk, convex_hull_image
)
from skimage.measure import label as sk_label, regionprops
from skimage import transform
from sklearn.cluster import KMeans

import torch
import segmentation_models_pytorch as smp

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("✅ Import'lar tamamlandı")

✅ Import'lar tamamlandı


## Hücre 3 — Pipeline Sınıfları

In [3]:
# ── SSA — Qi et al. 2013 ────────────────────────────────────
class SliceSelectionAlgorithm:
    BONE_HU_THRESHOLD      = 250
    SMALL_REGION_THRESH    = 200
    SKULL_CLOSE_THRESH     = 2
    TARGET_SLICE_COUNT     = 6
    MIN_INTRACRANIAL_RATIO = 0.10

    def select_slices(self, volume):
        scored = []
        for i in range(volume.shape[0]):
            score, valid = self._score_slice(volume[i])
            if valid:
                scored.append((i, score))
        if not scored:
            mid = volume.shape[0] // 2
            return list(range(max(0, mid-3), min(volume.shape[0], mid+3)))
        scored.sort(key=lambda x: x[1], reverse=True)
        return sorted(self._distribute([s[0] for s in scored], self.TARGET_SLICE_COUNT))

    def _score_slice(self, sl):
        bone = remove_small_objects(sl > self.BONE_HU_THRESHOLD,
                                    min_size=self.SMALL_REGION_THRESH)
        if not bone.any(): return 0.0, False
        lbl = sk_label(bone)
        if lbl.max() > self.SKULL_CLOSE_THRESH: return 0.0, False
        if not regionprops(lbl): return 0.0, False
        intracranial = ndimage.binary_fill_holes(bone) & ~bone
        if intracranial.sum() / sl.size < self.MIN_INTRACRANIAL_RATIO: return 0.0, False
        try:
            hull = convex_hull_image(intracranial)
            hull_area = hull.sum()
            if hull_area == 0: return 0.0, False
            return float(intracranial.sum() / hull_area), True
        except Exception:
            return 0.0, False

    def _distribute(self, cands, target):
        if len(cands) <= target: return cands
        step = len(cands) / target
        return [cands[int(i * step)] for i in range(target)]


# ── IML — Qi et al. 2013 ────────────────────────────────────
class IdealMidlineDetector:
    def detect(self, sl):
        skull = self._skull_mask(sl)
        if not skull.any(): return sl.shape[1] / 2.0, 0.0
        return self._exhaustive_search(skull)

    def _skull_mask(self, sl):
        bone = remove_small_objects(sl > 400, min_size=200)
        if not bone.any(): return bone
        lbl = sk_label(bone)
        regs = regionprops(lbl)
        if not regs: return bone
        return lbl == max(regs, key=lambda r: r.area).label

    def _exhaustive_search(self, skull, x_range=50, ang_range=10, ang_steps=21):
        H, W = skull.shape
        cx = W // 2
        xs = range(max(W//4, cx-x_range), min(3*W//4, cx+x_range))
        angles = np.linspace(-ang_range, ang_range, ang_steps)
        best_score, best_x = -1, float(cx)
        skull_f = skull.astype(np.float32)
        for ang in angles:
            rot = transform.rotate(skull_f, ang, center=(H//2, cx),
                                   preserve_range=True) > 0.5
            for x in xs:
                s = self._sym_score(rot, x)
                if s > best_score:
                    best_score, best_x = s, x
        return best_x, 0.0

    def _sym_score(self, mask, x):
        H, W = mask.shape
        left = mask[:, max(0, 2*x-W):x]
        right = mask[:, x:min(W, 2*x)]
        right_flip = np.fliplr(right)
        w = min(left.shape[1], right_flip.shape[1])
        if w == 0: return 0.0
        L = left[:, -w:]
        R = right_flip[:, :w]
        return (L & R).sum() / max((L | R).sum(), 1)

    def estimate_from_volume(self, vol, slices):
        return np.median([self.detect(vol[i])[0] for i in slices]), 0.0


# ── DML — Qi et al. 2013 ────────────────────────────────────
class DeformedMidlineDetector:
    CSF_HU_MIN = -10
    CSF_HU_MAX = 15
    MIN_VENTRICLE_AREA = 1000
    MAX_VENTRICLE_AREA = 30000
    MAX_IML_DIST_RATIO = 0.15

    def detect(self, sl, iml_x, pixel_spacing, unet_mask=None):
        brain = self._brain_region(sl)
        if unet_mask is not None:
            brain = brain & ~unet_mask.astype(bool)
        csf = self._csf(sl, brain)
        dml_x = self._from_csf(csf, sl.shape[1], iml_x)
        method = "septum_pellucidum"
        if dml_x is None:
            dml_x = self._centroid_fallback(sl, brain)
            method = "centroid"
        return dml_x, {"method": method, "csf_px": int(csf.sum())}

    def _brain_region(self, sl):
        tissue = (sl > -10) & (sl < 150) & ~(sl > 400)
        tissue = binary_closing(tissue, disk(3))
        lbl = sk_label(tissue)
        regs = regionprops(lbl)
        if not regs: return tissue
        return ndimage.binary_fill_holes(lbl == max(regs, key=lambda r: r.area).label)

    def _csf(self, sl, brain):
        csf = (sl >= self.CSF_HU_MIN) & (sl <= self.CSF_HU_MAX) & brain
        return remove_small_objects(csf, min_size=100)

    def _from_csf(self, csf, width, iml_x):
        if not csf.any(): return None
        regs = regionprops(sk_label(csf))
        regs = [r for r in regs
                if self.MIN_VENTRICLE_AREA <= r.area <= self.MAX_VENTRICLE_AREA]
        if not regs: return None
        max_dist = width * self.MAX_IML_DIST_RATIO
        regs = [r for r in regs if abs(r.centroid[1] - iml_x) <= max_dist]
        if not regs: return None
        left  = [r for r in regs if r.centroid[1] < iml_x]
        right = [r for r in regs if r.centroid[1] >= iml_x]
        if not left or not right:
            if len(regs) >= 2:
                top2 = sorted(regs, key=lambda r: r.area, reverse=True)[:2]
                xs = sorted([r.centroid[1] for r in top2])
                return (xs[0] + xs[1]) / 2
            return regs[0].centroid[1] if regs else None
        return (max(left, key=lambda r: r.area).centroid[1] +
                max(right, key=lambda r: r.area).centroid[1]) / 2

    def _centroid_fallback(self, sl, brain):
        if not brain.any(): return sl.shape[1] / 2.0
        return ndimage.center_of_mass(brain)[1]


# ── MLS Calculator — Qi et al. 2013 ─────────────────────────
class MidlineShiftCalculator:
    ROTTERDAM_THRESHOLD_MM = 5.0

    def from_volume(self, volume, slices, iml_x, px_spacing, unet_vol=None):
        dml_det = DeformedMidlineDetector()
        results = []
        for idx in slices:
            mask = unet_vol[idx] if unet_vol is not None else None
            dml_x, dbg = dml_det.detect(volume[idx], iml_x, px_spacing, mask)
            if dml_x is not None:
                diff_mm = abs(dml_x - iml_x) * px_spacing
                results.append({"slice_idx": idx, "dml_x": dml_x,
                                 "mls_mm": diff_mm, "debug": dbg})
        if not results:
            return {"mls_mm": 0.0, "iml_x": iml_x}
        sp = [r for r in results if r["debug"]["method"] == "septum_pellucidum"]
        vals = np.array([r["mls_mm"] for r in (sp if sp else results)])
        if len(vals) >= 3:
            med = np.median(vals)
            mad = np.median(np.abs(vals - med))
            vals = vals[vals <= med + 2 * mad]
        return {"mls_mm": round(float(np.mean(vals)), 2), "iml_x": iml_x,
                "slice_results": results}


# ── Basal Cistern — Toledo et al. 2021 ──────────────────────
class BasalCisternCalculator:
    CR_NORMAL_THRESHOLD  = 0.20
    CR_PARTIAL_THRESHOLD = 0.08
    HU_MIN = -5
    HU_MAX =  45
    KMEANS_INIT = np.array([[0.0], [25.0], [35.0]])
    MIDBRAIN_RANGE = (0.25, 0.42)

    def calculate_cr(self, volume, pixel_spacing=1.0, unet_vol=None):
        n = volume.shape[0]
        lo, hi = int(n * self.MIDBRAIN_RANGE[0]), int(n * self.MIDBRAIN_RANGE[1])
        best_idx, best_score = (lo+hi)//2, -1.0
        for i in range(lo, hi+1):
            sl = volume[i]
            score = float(((sl >= self.HU_MIN) & (sl <= self.HU_MAX) & ~(sl > 400)).sum())
            if score > best_score:
                best_score, best_idx = score, i

        sl = volume[best_idx]
        bone = sl > 400
        tissue = (sl > -50) & (sl < 200) & ~bone
        filled = ndimage.binary_fill_holes(tissue)
        lbl = sk_label(filled)
        regs = regionprops(lbl)
        brain_mask = (lbl == max(regs, key=lambda r: r.area).label) if regs else filled
        if unet_vol is not None:
            brain_mask = brain_mask & ~unet_vol[best_idx].astype(bool)

        hu_filtered = (sl >= self.HU_MIN) & (sl <= self.HU_MAX) & brain_mask
        if hu_filtered.sum() < 50:
            return {"compression_ratio": 0.0, "basal_score": 2,
                    "status": "absent (invalid)", "valid": False}

        hu_vals = sl[hu_filtered].reshape(-1, 1)
        km = KMeans(n_clusters=3, init=self.KMEANS_INIT,
                    n_init=1, max_iter=300, random_state=42)
        km_labels = km.fit_predict(hu_vals)
        centers = km.cluster_centers_.flatten()
        sorted_ids = np.argsort(centers)
        csf_id = sorted_ids[0]
        n_csf = int((km_labels == csf_id).sum())
        n_par = int(np.sum([km_labels == sorted_ids[j] for j in [1,2]]))
        cr = n_csf / n_par if n_par > 0 else 0.0

        if cr >= self.CR_NORMAL_THRESHOLD:
            basal_score, status = 0, "normal"
        elif cr >= self.CR_PARTIAL_THRESHOLD:
            basal_score, status = 1, "compressed"
        else:
            basal_score, status = 2, "absent"

        return {"compression_ratio": round(cr, 4), "basal_score": basal_score,
                "status": status, "valid": True}


# ── Rotterdam Score — Maas et al. 2005 ──────────────────────
class RotterdamScoreCalculator:
    _mortality = {2: 0.07, 3: 0.12, 4: 0.27, 5: 0.54, 6: 0.61}

    def calculate(self, mls_mm, basal=0, edh_absent=1, ivh_sah=0, hemorrhage_volume=0):
        total = 1 + basal + (1 if mls_mm >= 5.0 else 0) + edh_absent + ivh_sah
        total = min(6, max(2, total))
        risk = ("Low risk" if total <= 2 else "Moderate risk" if total == 3
                else "High risk" if total == 4 else "Very high risk")
        return {"rotterdam_score": total,
                "estimated_mortality": self._mortality.get(total, 0.5),
                "risk_category": risk}


print("✅ Pipeline sınıfları hazır")

✅ Pipeline sınıfları hazır


## Hücre 4 — U-Net Model

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")

model = smp.Unet(encoder_name="resnet34", in_channels=3, classes=1).to(DEVICE)

try:
    model.load_state_dict(torch.load(
        "/kaggle/input/datasets/rembengbal/unet-model2/unet_ct_ich_best.pth",
        map_location=DEVICE
    ))
    print("✅ U-Net yüklendi (Dice: 0.82)")
except Exception as e:
    print(f"⚠️  Model yüklenemedi: {e}")

model.eval()


def window_hu(img, center, width):
    mn, mx = center - width//2, center + width//2
    return (np.clip(img, mn, mx) - mn) / (mx - mn)


def predict_volume(volume, model, device):
    """volume: (D,H,W) → returns (D,H,W) binary mask"""
    unet_vol = np.zeros_like(volume, dtype=np.uint8)
    with torch.no_grad():
        for i in range(volume.shape[0]):
            sl = volume[i]
            c1 = window_hu(sl,  40,   80)
            c2 = window_hu(sl, 500, 3000)
            c3 = window_hu(sl, 175,  350)
            tensor = torch.tensor(
                np.stack([c1, c2, c3], axis=0).astype(np.float32)
            ).unsqueeze(0).to(device)
            prob = torch.sigmoid(model(tensor)).cpu().numpy()[0, 0]
            unet_vol[i] = (prob > 0.5).astype(np.uint8)
    return unet_vol


def classify_hemorrhage_by_rules(vol, unet_mask):
    """
    vol      : (H,W,D) slice-last
    unet_mask: (H,W,D) slice-last
    Referans: Wintermark et al. (2008), Lee et al. (2020)
    """
    H, W, D = vol.shape
    cx, cy = W//2, H//2
    edh_pixels = sah_ivh_pixels = 0
    for d in range(D):
        sl = vol[:, :, d]
        mask_sl = unet_mask[:, :, d]
        if mask_sl.sum() == 0: continue
        for (y, x) in np.argwhere(mask_sl > 0):
            hu = float(sl[y, x])
            is_peripheral = min(y, H-y, x, W-x) < H * 0.15
            is_central    = np.sqrt((x-cx)**2 + (y-cy)**2) < min(H,W) * 0.25
            if is_peripheral and 55 <= hu <= 90:
                edh_pixels += 1
            elif is_central and 40 <= hu <= 70:
                sah_ivh_pixels += 1
    return edh_pixels >= 200, sah_ivh_pixels >= 100


print("✅ U-Net ve kural sistemi hazır")

✅ Device: cuda


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

✅ U-Net yüklendi (Dice: 0.82)
✅ U-Net ve kural sistemi hazır


## Hücre 5 — Gradio Arayüzü

> Bu hücreyi çalıştırınca public URL gelir. Tarayıcıda aç, DICOM dosyalarını yükle.

In [5]:
import gradio as gr


def analyze_dicom_files(dicom_files):
    if not dicom_files:
        return "❌ Dosya yüklenmedi", None, "—", "—", "—", "—", "—", "—"

    try:
        # ── DICOM yükle ──────────────────────────────────
        slices = []
        for f in dicom_files:
            try:
                slices.append(pydicom.dcmread(f.name))
            except Exception:
                continue

        if not slices:
            return "❌ Geçerli DICOM yok", None, "—", "—", "—", "—", "—", "—"

        slices.sort(key=lambda s: float(
            getattr(s, "ImagePositionPatient", [0,0,0])[2]
        ))

        try:
            pixel_spacing = float(slices[0].PixelSpacing[0])
        except Exception:
            pixel_spacing = 1.0

        arrays = []
        for s in slices:
            try:
                arr = s.pixel_array.astype(np.float32)
                arr = arr * float(getattr(s, "RescaleSlope", 1)) + \
                      float(getattr(s, "RescaleIntercept", 0))
                arrays.append(arr)
            except Exception:
                continue

        if not arrays:
            return "❌ Pixel array okunamadı", None, "—", "—", "—", "—", "—", "—"

        vol = np.stack(arrays)   # (D, H, W)

        if vol.shape[0] < 5:
            return "❌ Yeterli slice yok (en az 5 gerekli)", None, "—", "—", "—", "—", "—", "—"

        # ── Binary U-Net ─────────────────────────────────
        unet = predict_volume(vol, model, DEVICE)   # (D,H,W)
        hemorrhage_volume = int(unet.sum())

        # ── Kural tabanlı sınıflandırma ──────────────────
        vol_hwd  = np.transpose(vol,  (1, 2, 0))
        unet_hwd = np.transpose(unet, (1, 2, 0))
        edh_present, sah_ivh_present = classify_hemorrhage_by_rules(vol_hwd, unet_hwd)

        # ── MLS + Basal ───────────────────────────────────
        ssa   = SliceSelectionAlgorithm()
        iml   = IdealMidlineDetector()
        mls_c = MidlineShiftCalculator()
        bas_c = BasalCisternCalculator()
        rot_c = RotterdamScoreCalculator()

        sel_slices   = ssa.select_slices(vol)
        iml_x, _     = iml.estimate_from_volume(vol, sel_slices)
        mls_result   = mls_c.from_volume(vol, sel_slices, iml_x, pixel_spacing, unet)
        basal_result = bas_c.calculate_cr(vol, pixel_spacing, unet)

        mls_mm       = mls_result["mls_mm"]
        cr           = basal_result["compression_ratio"]
        basal_score  = basal_result["basal_score"]
        basal_status = basal_result["status"]

        rotterdam = rot_c.calculate(
            mls_mm=mls_mm,
            basal=basal_score,
            edh_absent=0 if edh_present else 1,
            ivh_sah=1 if sah_ivh_present else 0,
            hemorrhage_volume=hemorrhage_volume
        )

        score     = rotterdam["rotterdam_score"]
        mortality = rotterdam["estimated_mortality"]
        risk      = rotterdam["risk_category"]

        # ── Görselleştirme — SSA'nın 6 slice'ı ──────────
        n_slices = len(sel_slices)
        fig, axes = plt.subplots(
            2, n_slices,
            figsize=(4 * n_slices, 9),
            facecolor="#0f1117"
        )
        fig.suptitle(
            f"SSA Tarafından Seçilen {n_slices} Slice  |  "
            f"Rotterdam: {score}  |  MLS: {mls_mm:.1f}mm",
            color="white", fontsize=12, y=1.01
        )

        # Tek slice gelirse axes boyutunu düzelt
        if n_slices == 1:
            axes = axes.reshape(2, 1)

        for ax in axes.flatten():
            ax.set_facecolor("#0f1117")

        # Slice bazlı MLS değerlerini al
        slice_mls = {}
        if "slice_results" in mls_result:
            for r in mls_result["slice_results"]:
                slice_mls[r["slice_idx"]] = r["mls_mm"]

        for col, sl_idx in enumerate(sel_slices):
            sl      = vol[sl_idx]
            mask_sl = unet[sl_idx]

            # ── Üst sıra: Ham BT ─────────────────────────
            axes[0, col].imshow(sl, cmap="gray", vmin=-50, vmax=150)
            axes[0, col].set_title(
                f"Slice {sl_idx}",
                color="white", fontsize=9
            )
            axes[0, col].axis("off")

            # ── Alt sıra: Kanama overlay + IML çizgisi ───
            axes[1, col].imshow(sl, cmap="gray", vmin=-50, vmax=150)

            # Kanama maskesi — kırmızı overlay
            if mask_sl.any():
                overlay = np.zeros((*mask_sl.shape, 4), dtype=np.float32)
                overlay[mask_sl > 0] = [1.0, 0.2, 0.2, 0.5]
                axes[1, col].imshow(overlay)

            # IML çizgisi — lime yeşil
            axes[1, col].axvline(
                iml_x, color="lime", lw=1.5,
                linestyle="--", alpha=0.8
            )

            # Slice bazlı MLS varsa başlığa yaz
            sl_mls = slice_mls.get(sl_idx)
            title  = f"MLS={sl_mls:.1f}mm" if sl_mls is not None else "IML"
            axes[1, col].set_title(title, color="#94a3b8", fontsize=8)
            axes[1, col].axis("off")

        plt.tight_layout(pad=0.5)
        img_path = "/tmp/rotterdam_result.png"
        plt.savefig(img_path, dpi=90, bbox_inches="tight", facecolor="#0f1117")
        plt.close()

        # ── Rapor ────────────────────────────────────────
        emoji = {2:"🟢", 3:"🟡", 4:"🟠", 5:"🔴", 6:"🔴"}.get(score, "⚪")
        summary = f"""## {emoji} Rotterdam CT Skoru: **{score}** / 6

**Tahmini Mortalite:** %{mortality*100:.0f}  
**Risk Kategorisi:** {risk}

---
### 📊 Bileşenler
| Bileşen | Değer | Puan |
|---|---|---|
| Midline Shift | {mls_mm:.1f} mm | {"1" if mls_mm >= 5 else "0"} |
| Basal Sistern | {basal_status} | {basal_score} |
| EDH | {"Var ✅" if edh_present else "Yok —"} | {"0" if edh_present else "1"} |
| SAH / IVH | {"Var ✅" if sah_ivh_present else "Yok —"} | {"1" if sah_ivh_present else "0"} |

---
### 🩸 U-Net
- Toplam kanama pikseli: **{hemorrhage_volume:,}**
- Basal CR: **{cr:.3f}**
- Seçilen slice'lar: **{sel_slices}**"""

        return (
            summary, img_path,
            str(score), f"%{mortality*100:.0f}",
            f"{mls_mm:.1f} mm", basal_status,
            "Var" if edh_present else "Yok",
            "Var" if sah_ivh_present else "Yok",
        )

    except Exception as e:
        import traceback
        return (
            f"❌ Hata: {str(e)}\n\n```\n{traceback.format_exc()}\n```",
            None, "—", "—", "—", "—", "—", "—"
        )


# ── Arayüz ──────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Base()) as demo:

    gr.HTML("""
    <div style="text-align:center; padding:20px 0">
        <h1 style="font-size:1.8rem; color:#e2e8f0">
            🧠 Rotterdam CT Score Calculator
        </h1>
        <p style="color:#64748b">
            Binary U-Net + HU Kural Sistemi | Qi et al. 2013 | Toledo et al. 2021
        </p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📂 DICOM Dosyası Yükle")
            gr.Markdown("Bir hastaya ait `.dcm` dosyalarını seçin (5mm CT Plain serisi önerilir).")
            dicom_input = gr.File(
                label="DICOM Dosyaları",
                file_count="multiple",
                file_types=[".dcm"],
                height=180,
            )
            analyze_btn = gr.Button("🔍 Analiz Et", variant="primary", size="lg")

            gr.Markdown("---")
            gr.Markdown("### 📋 Sonuçlar")
            with gr.Row():
                out_score     = gr.Textbox(label="Rotterdam Skoru")
                out_mortality = gr.Textbox(label="Mortalite")
            with gr.Row():
                out_mls   = gr.Textbox(label="MLS")
                out_basal = gr.Textbox(label="Basal Sistern")
            with gr.Row():
                out_edh    = gr.Textbox(label="EDH")
                out_sahivh = gr.Textbox(label="SAH/IVH")

        with gr.Column(scale=2):
            gr.Markdown("### 📊 Detaylı Rapor")
            out_summary = gr.Markdown("*Analiz bekleniyor...*")
            gr.Markdown("### 🖼 SSA Slice'ları (Ham BT + Kanama Maskesi)")
            out_image = gr.Image(
                label="6 Slice — Üst: Ham BT | Alt: Kanama + IML",
                type="filepath",
                height=420
            )

    gr.Markdown("""
---
### ℹ️ Rotterdam CT Skoru (Maas et al. 2005)
| Skor | Mortalite | Risk |
|---|---|---|
| 2 | %7  | 🟢 Düşük |
| 3 | %12 | 🟡 Orta |
| 4 | %27 | 🟠 Yüksek |
| 5 | %54 | 🔴 Çok Yüksek |
| 6 | %61 | 🔴 Çok Yüksek |
    """)

    analyze_btn.click(
        fn=analyze_dicom_files,
        inputs=[dicom_input],
        outputs=[out_summary, out_image, out_score, out_mortality,
                 out_mls, out_basal, out_edh, out_sahivh]
    )

print("🚀 Gradio başlatılıyor...")
demo.launch(share=True, debug=False, show_error=True)

🚀 Gradio başlatılıyor...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6c6d7743b760e2da78.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
